# GeoReady 04 · QA gate and outputs

The release gate from the Implementation Guide (p.5): zero mandatory-exclusion violations, grid
allocation within every known limit, unknown grid explicit, totals and units reconciling,
readiness weights summing to one, and the unit tests passing. Outputs are written only if the
gate passes.

In [1]:
import os, sys
from pathlib import Path
def _root():  # the geoready folder: from GEOREADY_ROOT, the working folder, or the QGIS Notebook plugin
    if os.environ.get("GEOREADY_ROOT"):
        return Path(os.environ["GEOREADY_ROOT"])
    starts = [Path.cwd()]
    try:
        from qgis.PyQt.QtCore import QSettings
        starts.append(Path(QSettings().value("QGISNotebook/last_directory", "") or "."))
    except ImportError:
        pass
    return next((p for s in starts for p in [s, *s.parents] if (p / "src" / "geoready").exists()), None)
ROOT = _root()
assert ROOT, "Open this notebook from inside the geoready folder, or set GEOREADY_ROOT"
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
if "qgis" in sys.modules:          # inside QGIS: draw figures off-screen
    import matplotlib; matplotlib.use("Agg")
from geoready.nb import show
CONFIG = "config/northern_cape_pv.yaml"

In [2]:
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from geoready.run import Run
from geoready import qa, grid, zones, nb
run = Run(CONFIG); cfg = run.cfg
z = pd.read_csv(run.path("zones_stage3.csv"))
fp = gpd.read_file(run.path("zones_footprint.gpkg"), layer="zones")
gdf = fp.merge(z, on="zone_id", how="inner")
with rasterio.open(run.path("eligibility.tif")) as ds:
    code = ds.read(1)
with rasterio.open(run.path("protected_mask.tif")) as ds:
    prot = ds.read(1) == 1

In [3]:
checks = []
# 1 mandatory exclusions, on the grid and on the polygons
n_bad = int(((code == 0) & prot).sum())
checks.append({"check": "mandatory exclusions (grid)", "passed": n_bad == 0, "detail": f"{n_bad} eligible cells touch protected areas"})
pa = gpd.read_file(cfg["vectors"], layer="protected").to_crs(gdf.crs)
pa = pa[pa.intersects(gdf.union_all().envelope)]
checks.append(qa.no_exclusion_violation(gdf, pa, tol_km2=0.01))
# 2 grid allocation, per scenario
chain = ["sub", "local", "supply"]
for sid, sc in cfg["grid"]["scenarios"].items():
    a = z.rename(columns={f"alloc_mw_{sid}": "alloc_mw"}).assign(grid_status="known", binding=z[f"binding_{sid}"])
    r = qa.allocation_within_limits(a, grid.limits_for(z, sc["use"]), chain)
    r["check"] += f" ({sid})"; checks.append(r)
# 3 unknown grid explicit
a = z.assign(alloc_mw=z["alloc_mw_published"], binding=z["binding_published"],
             grid_status=np.where(z["binding_published"].astype(str).str.startswith("unknown"), "unknown", "known"))
checks.append(qa.unknowns_explicit(a))
# 4 totals and units
cap = cfg["capacity"]
checks.append(qa.totals_reconcile(z, "eligible_km2", "capacity_mw", cap["density_mwdc_per_km2"] * cap["land_use_factor"]["base"]))
# 5 weights
checks.append(qa.weights_sum_to_one(cfg["readiness"]))
# 6 unit tests
checks.append(nb.run_tests("tests"))
report = qa.gate(checks)
show(pd.DataFrame(report["checks"]))

,check,passed,detail
0,mandatory exclusions (grid),True,0 eligible cells touch protected areas
1,mandatory exclusions,True,overlap 0.0000 km² (tolerance 0.01)
2,grid allocation (published),True,all limits respected
3,grid allocation (strengthened),True,all limits respected
4,grid allocation (unconstrained),True,all limits respected
5,unknown grid explicit,True,0 zones behind unknown headroom
6,units and totals,True,max difference 5.68e-14 MW
7,readiness weights,True,sum 1.000
8,unit tests,True,59 passed in 1.10s


In [4]:
assert report["passed"], "QA gate failed: outputs not written"
gpkg = run.path("geoready.gpkg")
gdf.to_file(gpkg, layer="zones", driver="GPKG")
run.add_output("geoready.gpkg", "candidate zones with capacity, cost, grid allocation and readiness by scenario")
z.to_csv(run.path("zones.csv"), index=False); run.add_output("zones.csv", "zone table (no geometry)")
for f in ["eligibility.tif", "supply_curve_published.csv", "supply_curve_strengthened.csv",
          "supply_curve_unconstrained.csv", "eligibility_summary.csv", "supply_curves.png"]:
    run.add_output(f)
# OSeMOSYS-style adapter table: capacity and cost bands by GCCA supply area
bands = pd.cut(z["lcoe_base"], [0, 775, 800, 825, 1e9], labels=["<775", "775-800", "800-825", ">825"])
adapter = (z.assign(band=bands).groupby(["supply_area", "band"], observed=True)
             .agg(tech_mw=("capacity_mw", "sum"), mw_strengthened=("alloc_mw_strengthened", "sum"),
                  mw_published=("alloc_mw_published", "sum"), lcoe_base_mean=("lcoe_base", "mean")).round(1).reset_index())
adapter.to_csv(run.path("adapter_regional_bands.csv"), index=False); run.add_output("adapter_regional_bands.csv")
record = run.finish(report)
print(run.run_id, "outputs:", list(record["outputs"]))

2026-11-26_nc-pv-v01_3d5fc633 outputs: ['geoready.gpkg', 'zones.csv', 'eligibility.tif', 'supply_curve_published.csv', 'supply_curve_strengthened.csv', 'supply_curve_unconstrained.csv', 'eligibility_summary.csv', 'supply_curves.png', 'adapter_regional_bands.csv']


## Open the results in QGIS

In plain Jupyter the next cell does nothing. Run inside QGIS (for example with the
**QGIS Notebook** plugin) and it adds the zones and the eligibility grid to your map, styled.

In [5]:
from geoready import qgis
qgis.show(run.path("geoready.gpkg"), rasters=[run.path("eligibility.tif")])

False